In [0]:
import json
import re
from datetime import datetime, timezone

sa = "stmaritimeks"
bronze_root = f"abfss://bronze@{sa}.dfs.core.windows.net"
landing_root = f"{bronze_root}/_landing_local"

ingest_date = datetime.now(timezone.utc).strftime("%Y-%m-%d")
run_id = f"backfill_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}"

def list_files_recursive(path):
    files = []
    for item in dbutils.fs.ls(path):
        if item.isDir():
            files.extend(list_files_recursive(item.path))
        elif item.name.endswith(".jsonl") and not item.name.endswith("_ALL.jsonl"):
            files.append(item)
    return files

landing_files = list_files_recursive(landing_root)
print(f"Found {len(landing_files)} chunk files in _landing_local")

world_pattern = re.compile(r"^(gap|fishing|loitering|encounter)s?_(\d{4}-\d{2})\.jsonl$")
port_pattern = re.compile(r"^([a-z_]+)_(\d{4}-\d{2})\.jsonl$")

manifest_records = []
moved_count = 0

for f in landing_files:
    fname = f.name
    now_iso = datetime.now(timezone.utc).isoformat()
    
    if fname == "vessels_enriched.jsonl":
        target_path = f"{bronze_root}/gfw/vessels/load_type=initial/ingest_date={ingest_date}/batch_0001.jsonl"
        entity = "vessels"
        dataset = "public-global-vessel-identity:latest"
        region = "world"
        month_str = "2025-01_to_2025-02"
    elif world_pattern.match(fname):
        m = world_pattern.match(fname)
        event_type, month_str = m.group(1), m.group(2)
        target_path = f"{bronze_root}/gfw/events/{event_type}/load_type=initial/ingest_date={ingest_date}/world_{month_str}.jsonl"
        entity = event_type
        dataset = f"public-global-{event_type}s-events:latest"
        region = "world"
    elif port_pattern.match(fname):
        m = port_pattern.match(fname)
        region, month_str = m.group(1), m.group(2)
        target_path = f"{bronze_root}/gfw/events/port_visit/load_type=initial/ingest_date={ingest_date}/{region}_{month_str}.jsonl"
        entity = "port_visit"
        dataset = "public-global-port-visits-events:latest"
    else:
        print(f"Skipping unrecognized file: {fname}")
        continue

    row_count = spark.read.text(f.path).count()
    dbutils.fs.mv(f.path, target_path)
    moved_count += 1
    print(f"Moved {fname} ({row_count} rows) -> {target_path}")

    manifest_records.append({
        "run_id": run_id,
        "entity": entity,
        "dataset": dataset,
        "region": region,
        "window_start": f"{month_str}-01" if len(month_str) == 7 else "2025-01-01",
        "window_end": f"{month_str}-28" if len(month_str) == 7 else "2025-02-28",
        "load_type": "initial",
        "source": "local_script_backfill",
        "rows_received": row_count,
        "file_path": target_path,
        "status": "SUCCESS",
        "retry_count": 0,
        "error_message": None,
        "started_at_utc": now_iso,
        "finished_at_utc": datetime.now(timezone.utc).isoformat(),
        "source_url": "https://gateway.api.globalfishingwatch.org/v3",
        "api_version": "v3"
    })

if manifest_records:
    manifest_path = f"{bronze_root}/_manifest/ingestion_manifest/{run_id}.jsonl"
    manifest_body = "\n".join(json.dumps(r) for r in manifest_records)
    dbutils.fs.put(manifest_path, manifest_body, True)
    print(f"\nDone! Moved {moved_count} files and wrote manifest to {manifest_path}")

watermark_path = f"{bronze_root}/_state/watermark.json"
watermark_data = {
    "last_loaded_month": "2025-02",
    "window_end": "2025-02-28",
    "updated_at_utc": datetime.now(timezone.utc).isoformat(),
    "run_id": run_id
}
dbutils.fs.put(watermark_path, json.dumps(watermark_data, indent=2), True)
print(f"Wrote initial watermark to {watermark_path}")